# Macro-Factor Return Forecasting and Portfolio Optimization using the Black-Litterman Model

## Introduction

This notebook implements a quantitative portfolio optimization framework based on the Black–Litterman model, which extends the classical Markowitz approach by incorporating subjective investor views in a Bayesian setting.

Specifically, this notebook:

- Uses the PyPortfolioOpt library to implement the Black–Litterman model
(see documentation: https://pyportfolioopt.readthedocs.io/en/latest/BlackLitterman.html
)

- Integrates a multiple linear regression model to predict expected asset returns based on macroeconomic factors, which are then translated into Black–Litterman views

- Performs backtesting on real stock and ETF data to evaluate portfolio behavior

**Motivation**

The classical Markowitz mean–variance framework is highly sensitive to return estimates, often resulting in unstable and unintuitive portfolio allocations.
The Black–Litterman model addresses this issue by combining implied market equilibrium returns with customized views and confidence levels, leading to more robust and practically implementable portfolios.

**Libraries used**

- PyPortfolioOpt — portfolio optimization and Black–Litterman implementation

- yfinance — historical asset price data

- scikit-learn — linear regression models

- pandas_datareader — macroeconomic data

- matplotlib / seaborn — data visualization

In [ ]:
!pip install PyPortfolioOpt yfinance pandas numpy matplotlib seaborn scikit-learn pandas_datareader vectorbt numba

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pypfopt import BlackLittermanModel, EfficientFrontier, expected_returns, risk_models, black_litterman, plotting
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas_datareader.data as web
from datetime import datetime
import vectorbt as vbt
from vectorbt.portfolio.nb import order_nb, sort_call_seq_nb
from vectorbt.portfolio.enums import SizeType, Direction
from numba import njit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import requests
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print('Imports done !')

## Theory and Reminders

### Markowitz Model (Modern Portfolio Theory)

The classical portfolio optimization framework is defined as follows:

- **Expected portfolio return**:  
  $$ R_p = w^\top \mu $$

- **Portfolio risk (variance)**:  
  $$ \sigma_p^2 = w^\top \Sigma w $$

- **Optimization problem (minimum variance)**:
  $$
  \min_w \; w^\top \Sigma w
  \quad \text{s.t.} \quad
  w^\top \mu = R_{\text{target}}, \quad
  w^\top \mathbf{1} = 1, \quad
  w \geq 0
  $$

Solving this problem for different target returns traces out the **efficient frontier**.

---

### Black–Litterman Model

The Black–Litterman model is a Bayesian extension of the Markowitz framework that incorporates **subjective investor views**.

- **Implied equilibrium returns (prior)**:
  $$
  \pi = \delta \Sigma w_m
  $$

  where:
  * δ is the risk aversion parameter
  * wₘ are the market capitalization weights

- **Investor views**:

  * P ∈ ℝ^(K×N): view matrix  
  * Q ∈ ℝ^K: view returns  
  * Ω ∈ ℝ^(K×K): diagonal covariance matrix of view uncertainty


- **Posterior expected returns**:
  $$
  \mu_{BL} =
  \left[(\tau \Sigma)^{-1} + P^\top \Omega^{-1} P\right]^{-1}
  \left[(\tau \Sigma)^{-1} \pi + P^\top \Omega^{-1} Q\right]
  $$

- **Posterior covariance (simplified form)**:
  $$
  \Sigma_{BL} = \Sigma + \left[(\tau \Sigma)^{-1} + P^\top \Omega^{-1} P\right]^{-1}
  $$
  with $\tau$ representing uncertainty on prior, usually $\in [0.01, 0.05]$

- **Going back to Markowitz now**
  * We go back to Markowitz optimization but using as input, the **BL** returns and covariance, the purpose being to **maximize** the ***Sharpe Ratio*** of the portfolio or to **minimize** its variance
  $$\underset {\omega}{max} \left( \omega^T \mu_{BL} - \frac {\delta} {2} \omega^T \Sigma_{BL} \omega\right)$$
  * We will use the pypfopt library for that

---

### Linear Regression for expected returns
For each asset $i$, we model the expected returns as:

$$
R_{i,t+1} = \alpha_i + \beta_{i,1} F^1_t + \beta_{i,2} F^2_t + \dots + \beta_{i,k} F^k_t + \varepsilon_{i,t+1}
$$

where:

- $R_{i,t+1}$ is the return of asset $i$ at time $t+1$,
- $F^j_t$ are the observed macroeconomic factors at time $t$,
- $\beta_{i,j}$ measure the sensitivity (exposure) of asset $i$ to factor $j$,
- $\alpha_i$ is the intercept (asset-specific alpha),
- $\varepsilon_{i,t+1}$ is an idiosyncratic error term assumed to be normally distributed.

### To visualize the Black-Litterman flow:

Market priors → Views (predicted via ML) → Adjusted returns → Optimization

## Global Configuration

All parameters are centralised here. Modify this cell only to change the experiment settings.

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION — edit only this cell to run experiments
# ============================================================

# Universe
TICKERS = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOG', 'META', 'AVGO', 'TSLA', 'BRK-B', 'WMT',
           'LLY', 'JPM', 'XOM', 'V', 'JNJ', 'MU', 'COST', 'ORCL', 'MA']
MARKET_TICKER = 'SPY'          # Benchmark / market proxy

# Date range
START_DATE = '2021-01-10'

# Risk & optimisation
RISK_FREE_RATE = 0.02          # Annual
COV_METHOD     = 'ledoit_wolf_constant_correlation'
TAU            = 0.05          # BL prior uncertainty
MOMENTUM_K     = 0.05          # Momentum scaling coefficient
RIDGE_ALPHA    = 1.0           # Ridge regularisation strength

# Backtesting
REBALANCE_EVERY = 50           # Trading days between rebalances

# Views method: 'momentum' | 'multi_linear_regression'
METHOD_VIEWS = 'momentum'

print('Configuration loaded.')

## Data Collection and Preparation

In [ ]:
def get_sp500_tickers() -> list[str]:
    """Scrape S&P 500 constituents from Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    table = pd.read_html(StringIO(r.text))[0]
    return table['Symbol'].str.replace('.', '-', regex=False).tolist()


def get_market_caps(tickers: list[str]) -> tuple[dict, dict]:
    """
    Return (cap_weights, market_caps) for the given tickers.
    Tickers with unavailable market cap are skipped gracefully.
    """
    market_caps = {}
    for ticker in tickers:
        cap = yf.Ticker(ticker).info.get('marketCap')
        if cap is not None:
            market_caps[ticker] = cap

    total = sum(market_caps.values())
    weights = {t: cap / total for t, cap in market_caps.items()}
    return weights, market_caps


def get_prices(tickers: list[str] | str, start_date: str = '2015-01-01') -> pd.DataFrame:
    """
    Download adjusted close prices from Yahoo Finance.
    Always returns a DataFrame (even for a single ticker).
    """
    end_date = datetime.today().strftime('%Y-%m-%d')
    raw = yf.download(tickers, start=start_date, end=end_date, progress=False, auto_adjust=True)

    # yfinance >= 0.2 returns a MultiIndex when multiple tickers are passed
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close']
    else:
        prices = raw[['Close']]  # single ticker — keep as DataFrame
        if isinstance(tickers, str):
            prices.columns = [tickers]

    # Drop timezone info for consistency
    if prices.index.tz is not None:
        prices.index = prices.index.tz_localize(None)

    return prices

## Black-Litterman Implementation
Once we understand the theory behind the model, the true challenge is to evaluate correctly the different parameters of the model, which are :
- $\pi$ : The equilibrium prior that depends on 
    - $\delta$ : The risk aversion, **evaluated using Sharpe Ratio Method**
    - $\Sigma$ : The covariance matrix, evaluated using :
        - Sample covariance method (<span style="color:green">simple, non biased </span> but <span style="color:red">noisy on small samples, Singular if T \lt N </span>) $$\Sigma = \frac{1}{T-1} \sum_{t=1}^{T} (r_t - \bar{r})(r_t - \bar{r})^T$$
        
        - Semicovariance method (<span style="color:green">Captures only downside risk, Coherent with risk aversion </span> but <span style="color:red">Upside information is lost and statistically less stable </span>) $$\Sigma_{semi} = \frac{1}{T} \sum_{t: r_t < \bar{r}} (r_t - \bar{r})(r_t - \bar{r})^T$$
        
        - Exponential Covariance, that gives more weight to recent observations (<span style="color:green">reacts to regime changes </span> but $\lambda$ <span style="color:red"> choice is arbirary </span>) $$\Sigma_{exp} = \frac{\sum_{t} \lambda^{T-t} (r_t - \bar{r})(r_t - \bar{r})^T}{\sum_{t} \lambda^{T-t}}$$

            |$\lambda$|Interprétation|
            |---|---|
            | $\lambda \to 1$ | All observations carry the same weight. |
            | $\lambda \to 0$ | Only recent observations matter |
            | $\lambda = 0.94$ | Standard RiskMetrics JP Morgan |

        - Ledoit-Wolf methods (shrinkage) $$\Sigma_{LW} = (1 - \alpha) \cdot \Sigma_{sample} + \alpha \cdot F$$ 
            ```
                Σ_sample  →  Noisy but faithful to the data
                F         →  structured, stable but biased target
                α         →  intensity of shrinkage (optimal, calculated analytically)
            ```
            > Ledoit & Wolf (2004) find $\alpha$ **optimal analytically** → no calibration needed
            1. ***Ledoit-Wolf with constant variance*** (<span style="color:green"> simple, stable </span> but <span style="color:red"> with the strong hypothesis **all variances are equal** </span>), $F$ which is a matrix with constant variance $$F_{ij} = \begin{cases} \bar{\sigma}^2 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases}$$
            2. ***Ledoit-Wolf single factor*** uses a factor method to value $F$ (like CAPM) (<span style="color:green"> economically motivated, good compromise bias/variance </span> but <span style="color:red"> assume that a factor alone explain the correlations</span>) $$F = \beta \beta^T \sigma_m^2 + D$$
            where $D$ is a diagonal matrix with specific variances
            3. ***Ledoit-Wolf with constant correlation*** (<span style="color:green"> keeps individual true variances, better compromise of the LW family </span> but <span style="color:red"> assume a constant correlation, which is generally incorrect </span>), $F$ is assuming an identical correlation between all the assets $$F_{ij} = \begin{cases} \sigma_i^2 & \text{if } i = j \\ \bar{\rho} \cdot \sigma_i \sigma_j & \text{if } i \neq j \end{cases}$$
        - Oracle Approximating (<span style="color:green"> Theoritically optimal </span> but <span style="color:red"> complex and computationally costful </span>):
            - Method of **Chen, Wiesel, Eldar & Hero (2010)** :
                - Find the matrix that minimizes the **expected quadratic loss**
                - \"Oracle\" because it approximates what a perfect estimator would do

- The investor views that depends on :
    - $P$ : Matrix representing the assets on which the investor has an opinion, each line is an opinion :
        - The sum of the row elements equals to 1 if it is an absolute view (*Asset 1 will have a performance of $x\%$*)
        - The sum of the row elements equals to 0 if it is a relative view (*Asset 1 will overperform Asset 2 and Asset 3 by $y\%$*)
        - Result $$P = \begin{pmatrix} 1 & 0 & 0 \\ 1 & -0.5 & -0.5 \end{pmatrix}$$
    - $Q$ : Is the matrix containing the values of these performances, in our exemple it would be $$Q = \begin{pmatrix} x\% \\ y\% \end{pmatrix}$$
    - $\Omega$ : Is the confidence matrix $$\Omega = \begin{pmatrix} \omega_1 & 0 \\ 0 & \omega_2 \end{pmatrix}$$ with $$\begin{aligned} &\omega \rightarrow 0 : \text{Total confidence in the view} \\ &\omega \rightarrow \infty : \text{View ignored}\end{aligned}$$
        - Calibrated using one of the following methods :
            - **Black-Litterman Method** : $\omega_i = \tau \cdot P_i \Sigma P_i^T$
            - **He & Litterman Method** : $\omega_i = \frac {1-c} {c} \cdot \tau \cdot P_i \Sigma P_i^T \text{ with } c \in [0,1]$
            - **Regression Method (Our Approach)** : $\omega_i = \text{Var}(\epsilon_i)$ (MSE of the macro-economic regression)


## Views
Now that we have calculated the prior and the covariance matrix, we will work on getting the investor views, for that, we wil implement different methods

### Momentum method
The main idea is "Assets that have performed well recently will continue to outperform"
This idea was documented by **Jegadeesh & Titman (1993)**, that shows that buying winners and selling losers on 3-12 months generates abnormal returns \
The theory behind
|Phenomenon|Description|
|---|---|
|Underreaction|Investors integrates slowly new informations|
|Herding|Investors follow trends|
|Disposition effect|Winners are sold too early, losers too late|

Classic signal : $$Mom_i = \frac {P_{t-1}}{P_{t-12}} - 1$$ Which is the momentum at time $i$ for the asset
>We exclude the last month (t-1) to avoid short-term reversal

To get our $Q_i$ :
1. We calculate the momentum of the asset : $$Mom_i = r_i(t-12, t-1) = \frac {P_{t-1}}{P_{t-12}} - 1$$ with $r_i$ return of asset $i$ (between month $t-1$ and month $t-12$)
2. We normalize the signal, to get the outperformers and underperformers. With the formula $$z_i = \frac {Mom_i-Mom}{\sigma_{Mom}}$$ with $Mom$ the mean of all the momentums, and $\sigma_{Mom}$ the standard deviation
3. We use the z-score obtained to calculate the view $Q_i$ : 
    1. First approach : $$Q_i = z_i * k$$ with k a coefficient
        - At first we will take $k = 0.05$
        - We will then calculate it as $k = \sigma_i \times \phi$ with $\phi$ the sharpe ratio wanted
        - Then we will use the information ratio approach (Grinold) with $k = \sigma_{residual} \times IC$ ($IC$ being the historical correlation between momentum scores and futur returns)
    2. Second approach (with only relative views) : 
        - Each line sum is 0 ($\sum p_i = 0$)
        - We select n top (with $p_i = \frac 1n$) and n bottom (with $p_i = - \frac 1n$) assets (performance wise)
        - We calculate $Q_i = (Z_{top} - Z_{bottom}) \times k$ (with Z_{top/bottom} the mean of Z-score of the top/bottom assets)

To get our $\Omega$ :
1. For the first approach, we will use the Black-Litterman method seen before  $\omega_i = \tau \cdot P_i \Sigma P_i^T$
2. For the second approach, we will use the historical variance of the difference between top and bottom assets returns $\omega_i = Var(R_{top} - R_{bottom})$

In [ ]:
def momentum(
    prices: pd.DataFrame,
    market_prior: pd.Series,
    sigma: pd.DataFrame,
    k: float = MOMENTUM_K,
    tau: float = TAU,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate Black-Litterman views using the momentum signal.

    Skip the last 21 days (≈1 month) to avoid short-term reversal
    (Jegadeesh & Titman, 1993).

    Returns:
        P      : (N x N) identity view matrix (absolute views)
        Q      : (N,) array of view returns
        Omega  : (N x N) diagonal uncertainty matrix
    """
    SKIP_DAYS = 21
    LOOKBACK  = 252  # 1 trading year

    start_idx = -LOOKBACK if len(prices) >= LOOKBACK else 0
    mom = prices.iloc[-SKIP_DAYS] / prices.iloc[start_idx] - 1

    z_scores = (mom - mom.mean()) / mom.std()
    Q = market_prior.values + z_scores.values * k

    P     = np.eye(len(prices.columns))
    Omega = tau * P @ sigma.values @ P.T

    return P, Q, Omega

### Linear Regression

### Multiple Linear Regression for Return Prediction

In this section, we employ a multiple linear regression model to forecast asset returns based on macroeconomic factors. This approach allows us to generate data-driven *views* on expected returns, which will be incorporated into the Black–Litterman framework.

For each asset $i$, we model the expected returns as:

$$
R_{i,t+1} = \alpha_i + \beta_{i,1} F^1_t + \beta_{i,2} F^2_t + \dots + \beta_{i,k} F^k_t + \varepsilon_{i,t+1}
$$

where:

- $R_{i,t+1}$ is the return of asset $i$ at time $t+1$,
- $F^j_t$ are the observed macroeconomic factors at time $t$,
- $\beta_{i,j}$ measure the sensitivity (exposure) of asset $i$ to factor $j$,
- $\alpha_i$ is the intercept (asset-specific alpha),
- $\varepsilon_{i,t+1}$ is an idiosyncratic error term assumed to be normally distributed.

The predicted returns ($\hat{R}_{i,t+1}$) from this model serve as absolute views ($Q$) in Black–Litterman, enhancing the equilibrium priors with empirical insights. We fit separate regressions for each asset to capture idiosyncratic sensitivities.

### Macro-Factor Selection

To forecast asset returns effectively, we select a parsimonious set of macroeconomic factors that are economically motivated, empirically validated, and publicly available. The goal is to capture key drivers of financial markets while minimizing model complexity, overfitting, and multicollinearity.

After analysis, we restrict the model to **four key macro factors**, each representing a distinct economic dimension. This reduction (from an initial consideration of five) avoids potential multicollinearity issues—e.g., between changes in long-term yields and yield curve slope—ensuring more stable and interpretable coefficients. Factors are sourced from reliable public databases like FRED (Federal Reserve Economic Data) and processed (e.g., differenced for stationarity where needed).

#### Selected Factors

1. **$\Delta$ 10Y Treasury Yield**  
   Measures changes in the 10-year U.S. Treasury yield, capturing shifts in long-term discount rates and valuation effects. This factor is particularly relevant for equity markets and growth stocks, as rising yields can compress multiples (e.g., as in Cochrane, 2008, on equity risk premiums).  
   Source: FRED (`GS10`); computed as first differences for stationarity.

2. **Inflation Surprise (CPI)**  
   Quantifies unexpected inflation shocks (actual CPI minus consensus forecast), which influence monetary policy expectations and real returns. High surprises can erode purchasing power and trigger rate hikes, impacting asset classes differently (e.g., Ang et al., 2008, on inflation hedging).  
   Source: FRED (`CPIAUCSL`) with approximations for surprises via lags or external forecasts.

3. **$\Delta$ VIX (Implied Volatility Index)**  
   Tracks changes in the CBOE Volatility Index, serving as a proxy for market risk aversion and uncertainty. Spikes in VIX often signal stress periods with negative equity returns (e.g., Carr & Wu, 2009, on volatility risk premium).  
   Source: FRED (`VIXCLS`); differenced to focus on shocks rather than levels.

4. **PMI Composite Index**  
   A forward-looking gauge of economic activity, blending manufacturing and services sectors. Higher PMI signals growth momentum, positively correlating with corporate earnings and stock returns (e.g., widely used in business cycle models by firms like JPMorgan).  
   Source: FRED (`NAPM` for ISM PMI) or equivalents; $z$-scored for normalization.

#### Why These Factors?

- **Economic Interpretation**: Each factor maps to a unique channel—interest rates (valuation), inflation (policy shocks), volatility (risk sentiment), and activity (growth)—providing clear insights into return drivers.

- **Distinctiveness and Robustness**: They are relatively orthogonal (low pairwise correlations, verified via heatmap in code), reducing multicollinearity risks. $VIF$ (Variance Inflation Factor) checks confirm stability.

- **Empirical Support**: Grounded in academic and practitioner literature (e.g., Fama–French extensions with macro factors; AQR's macro timing models).

- **Practicality**: All are freely accessible, updated frequently, and suitable for a quantitative project at student level. The limited set (4 factors) ensures model parsimony, with better out-of-sample performance than overparameterized alternatives.

- **Rationale for Reduction**: An initial set of five included yield curve slope, but it was dropped due to moderate correlation (around $0.5$) with $\Delta$ 10Y Yield, prioritizing interpretability and avoiding inflated variances in estimates.

This factor framework forms a robust basis for conditional expected returns, bridging macroeconomics and portfolio optimization.

In [ ]:
# FRED series used as macro factors
MACRO_SERIES: dict[str, str] = {
    'GS10':     '10Y_Yield',
    'CPIAUCSL': 'CPI',
    'VIXCLS':   'VIX',
    'INDPRO':   'IP',
}


def _fetch_macro(start_date: str, end_date: str) -> pd.DataFrame:
    """Download and forward-fill all FRED macro series."""
    frames = []
    for code, name in MACRO_SERIES.items():
        s = web.DataReader(code, 'fred', start_date, end_date)
        s.columns = [name]
        frames.append(s)
    return pd.concat(frames, axis=1).ffill()


def _stationarise_macro(macro_df: pd.DataFrame) -> pd.DataFrame:
    """
    Resample to month-end and apply stationarity transforms:
    - First-difference for rate/level series (10Y Yield, VIX)
    - Percentage change for index series (CPI, Industrial Production)
    """
    raw = macro_df.resample('ME').last()
    st = pd.DataFrame(index=raw.index)
    st['10Y_Diff'] = raw['10Y_Yield'].diff()
    st['VIX_Diff'] = raw['VIX'].diff()
    st['CPI_Ret']  = raw['CPI'].pct_change()
    st['IP_Ret']   = raw['IP'].pct_change()
    return st.dropna()


def get_cleaned_data(ticker: str, start_date: str = '2015-01-01') -> tuple[pd.DataFrame, pd.Series]:
    """
    Build the feature matrix X and target vector y for one ticker.

    Features are the 4 stationary macro factors LAGGED by 1 month
    to prevent look-ahead bias. Target is the compounded monthly return.

    Returns:
        X : (T x 4) DataFrame of lagged macro features (not yet scaled)
        y : (T,) Series of monthly asset returns
    """
    end_date = datetime.today().strftime('%Y-%m-%d')

    macro_monthly = _stationarise_macro(_fetch_macro(start_date, end_date))

    prices = get_prices(ticker, start_date).squeeze()  # Series
    asset_ret = (
        prices.pct_change()
               .resample('ME')
               .apply(lambda x: (1 + x).prod() - 1)
               .rename('Asset_Return')
    )

    merged = pd.concat([asset_ret, macro_monthly], axis=1).dropna()

    # Lag features by 1 month — shift AFTER merge to preserve alignment
    X = merged.drop(columns=['Asset_Return']).shift(1).dropna()
    y = merged['Asset_Return'].loc[X.index]

    return X, y


# Quick sanity check
try:
    _X, _y = get_cleaned_data('SPY')
    print(f'✅ get_cleaned_data OK  — X: {_X.shape}, y: {_y.shape}')
    print(_X.head(3))
except Exception as e:
    print(f'❌ Error: {e}')

In [ ]:
def multi_linear_regression(
    tickers: list[str],
    ridge_alpha: float = RIDGE_ALPHA,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fit a Ridge regression per ticker on macro factors and return
    the Black-Litterman (P, Q, Omega) triplet.

    - P     : identity matrix (absolute views)
    - Q     : predicted monthly return for each ticker
    - Omega : diagonal matrix where Ω_ii = MSE_i (regression residual variance)
    """
    N = len(tickers)
    P               = np.eye(N)
    q_views         = []
    omega_variances = []

    print('-' * 60)
    print('🚀 STARTING MACRO MODEL TRAINING')
    print('-' * 60)

    for ticker in tickers:
        X, y = get_cleaned_data(ticker)

        # Chronological split — never shuffle time series
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, shuffle=False
        )

        # Fit scaler on train only to prevent data leakage
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        model = Ridge(alpha=ridge_alpha)
        model.fit(X_train_sc, y_train)

        y_pred = model.predict(X_test_sc)
        mse    = mean_squared_error(y_test, y_pred)
        r2     = r2_score(y_test, y_pred)

        # Predict next-period return using latest available macro data
        latest_scaled    = scaler.transform(X.iloc[[-1]])
        expected_return  = float(model.predict(latest_scaled)[0])

        q_views.append(expected_return)
        omega_variances.append(mse)

        print(f'[{ticker:6s}] R²={r2:7.4f} | MSE={mse:.6f} | Q={expected_return:+.4%}')

    Q     = np.array(q_views)
    Omega = np.diag(omega_variances)

    print('-' * 60)
    print('✅ P, Q, Omega matrices generated.')

    return P, Q, Omega

### Factors method (Fama-French)
> ⚠️ To be implemented in a future version.

In [ ]:
_VALID_VIEW_METHODS = ('momentum', 'multi_linear_regression')


def get_views(
    tickers:      list[str],
    prices:       pd.DataFrame,
    market_prior: pd.Series,
    sigma:        pd.DataFrame,
    method:       str   = 'momentum',
    k:            float = MOMENTUM_K,
    tau:          float = TAU,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Dispatcher: route to the requested view-generation method.

    Supported methods: 'momentum', 'multi_linear_regression'
    """
    if method == 'momentum':
        return momentum(prices, market_prior=market_prior, sigma=sigma, k=k, tau=tau)
    elif method == 'multi_linear_regression':
        return multi_linear_regression(tickers)
    else:
        raise ValueError(
            f"Unknown view method '{method}'. ",
            f"Valid options: {_VALID_VIEW_METHODS}"
        )

In [ ]:
def _compute_sigma(prices: pd.DataFrame, cov_method: str) -> pd.DataFrame:
    """Compute the annualised covariance matrix using the chosen estimator."""
    supported = {
        'sample_cov', 'semicovariance', 'exp_cov',
        'ledoit_wolf', 'ledoit_wolf_constant_variance',
        'ledoit_wolf_single_factor', 'ledoit_wolf_constant_correlation',
        'oracle_approximating',
    }
    if cov_method not in supported:
        raise ValueError(f"Unknown cov_method '{cov_method}'. Supported: {supported}")
    return risk_models.risk_matrix(prices, method=cov_method, frequency=12)


def black_litterman_opt(
    tickers:        list[str],
    all_prices:     pd.DataFrame,
    market_ticker:  str   = MARKET_TICKER,
    cov_method:     str   = COV_METHOD,
    risk_free_rate: float = RISK_FREE_RATE,
    method_views:   str   = METHOD_VIEWS,
) -> dict[str, float]:
    """
    Run the full Black-Litterman optimisation pipeline.

    Parameters
    ----------
    tickers        : list of asset tickers (must match columns of all_prices[:-1])
    all_prices     : DataFrame with asset prices in the first N columns
                     and the market index in the last column
    market_ticker  : ticker symbol of the market index column
    cov_method     : covariance estimator (see _compute_sigma for options)
    risk_free_rate : annual risk-free rate used in Sharpe maximisation
    method_views   : 'momentum' or 'multi_linear_regression'

    Returns
    -------
    clean_weights : dict mapping ticker → portfolio weight
    """
    prices        = all_prices.iloc[:, :-1]
    market_prices = all_prices.iloc[:, -1]   # FIX: was using undefined 'market' variable

    _, market_caps = get_market_caps(tickers)

    sigma = _compute_sigma(prices, cov_method)

    delta = black_litterman.market_implied_risk_aversion(
        market_prices, frequency=12, risk_free_rate=risk_free_rate
    )

    market_prior = black_litterman.market_implied_prior_returns(
        market_caps, delta, sigma, risk_free_rate=risk_free_rate / 12
    )

    P, Q, Omega = get_views(tickers, prices, market_prior, sigma, method=method_views)

    # For regression views Q is already monthly — add to monthly prior
    if method_views == 'multi_linear_regression':
        Q = Q.flatten() + market_prior.values

    bl = BlackLittermanModel(sigma, pi=market_prior, P=P, Q=Q, omega=Omega)
    posterior_returns = bl.bl_returns()
    posterior_cov     = bl.bl_cov()

    ef = EfficientFrontier(posterior_returns, posterior_cov)
    ef.max_sharpe(risk_free_rate=risk_free_rate / 12)
    return ef.clean_weights()


def markowitz_opt(
    tickers:        list[str],
    all_prices:     pd.DataFrame,
    market_ticker:  str   = MARKET_TICKER,   # unused, kept for uniform signature
    cov_method:     str   = COV_METHOD,       # unused, kept for uniform signature
    risk_free_rate: float = RISK_FREE_RATE,
    method_views:   str   = METHOD_VIEWS,     # unused, kept for uniform signature
) -> dict[str, float]:
    """
    Classical mean-variance optimisation (max Sharpe).
    Signature mirrors black_litterman_opt for drop-in use in backtesting().
    """
    prices = all_prices.iloc[:, :-1]
    mu     = expected_returns.mean_historical_return(prices)
    sigma  = risk_models.sample_cov(prices)
    ef = EfficientFrontier(mu, sigma)
    ef.max_sharpe(risk_free_rate=risk_free_rate)
    return ef.clean_weights()

# Backtesting

In [ ]:
# VectorBT global settings
vbt.settings.array_wrapper['freq']          = 'days'
vbt.settings.returns['year_freq']            = '252 days'
vbt.settings.portfolio['seed']               = 42
vbt.settings.portfolio.stats['incl_unrealized'] = True


@njit
def pre_sim_func_nb(c, every_nth: int):
    """Activate only every N-th day for rebalancing; skip all others."""
    c.segment_mask[:, :] = False
    c.segment_mask[every_nth::every_nth, :] = True
    return ()


def pre_segment_func_nb(c, find_weights_fn, history_len, tickers, all_prices,
                         cov_method, risk_free_rate, start_date, method_views):
    """
    Called on every active (rebalancing) day.
    Computes optimal weights and informs VectorBT of the order sequence.

    NOTE: Runs as pure Python (not Numba-compiled) because it calls
    yfinance, sklearn, and PyPortfolioOpt which are incompatible with Numba.
    """
    if history_len == -1:
        close = c.close[:c.i, c.from_col:c.to_col]
    else:
        if c.i - history_len <= 0:
            return (np.full(c.group_len, np.nan),)
        close = c.close[c.i - history_len:c.i, c.from_col:c.to_col]

    weights_dict = find_weights_fn(tickers, all_prices, cov_method, risk_free_rate, start_date, method_views)
    weights      = np.array([weights_dict.get(t, 0.0) for t in tickers])

    size_type       = SizeType.TargetPercent
    direction       = Direction.LongOnly
    order_value_out = np.empty(c.group_len, dtype=np.float64)
    for k in range(c.group_len):
        c.last_val_price[c.from_col + k] = c.close[c.i, c.from_col + k]
    sort_call_seq_nb(c, weights, size_type, direction, order_value_out)

    return (weights,)


@njit
def order_func_nb(c, weights):
    """Place a TargetPercent order for the current asset."""
    col_i = c.call_seq_now[c.call_idx]
    return order_nb(
        weights[col_i],
        c.close[c.i, c.col],
        size_type=SizeType.TargetPercent,
    )


def backtesting(
    tickers:        list[str]   = TICKERS,
    market_ticker:  str         = MARKET_TICKER,
    cov_method:     str         = COV_METHOD,
    risk_free_rate: float       = RISK_FREE_RATE,
    start_date:     str         = START_DATE,
    method_views:   str         = METHOD_VIEWS,
    every_nth:      int         = REBALANCE_EVERY,
    model                       = black_litterman_opt,
) -> vbt.Portfolio:
    """
    Run a full backtest with periodic rebalancing.

    The market index (last column of all_prices) is used both for:
    - Computing BL equilibrium priors (delta, marketPrior)
    - Providing the SPY benchmark for performance comparison
    """
    prices        = get_prices(tickers, start_date).dropna(how='all').ffill()
    market_prices = get_prices(market_ticker, start_date).dropna(how='all').ffill()
    all_prices    = prices.join(market_prices, how='inner')
    active_tickers = list(prices.columns)

    print(f'Universe: {active_tickers}')
    print(f'Period  : {all_prices.index[0].date()} → {all_prices.index[-1].date()}')

    pf = vbt.Portfolio.from_order_func(
        all_prices,
        order_func_nb,
        pre_sim_func_nb   = pre_sim_func_nb,
        pre_sim_args      = (every_nth,),
        pre_segment_func_nb = pre_segment_func_nb,   # pure Python — no .py_func needed
        pre_segment_args  = (model, -1, active_tickers, all_prices,
                              cov_method, risk_free_rate, start_date, method_views),
        cash_sharing = True,
        group_by     = True,
        use_numba    = False,
    )

    if len(pf.orders.records) == 0:
        print('⚠️  No filled orders — check your configuration.')
        return pf

    first_trade_idx = int(pf.orders.records['idx'].min())

    mkt          = all_prices.iloc[:, -1].astype(float)
    bench_value  = mkt / mkt.iloc[first_trade_idx]
    bench_value.iloc[:first_trade_idx] = np.nan
    benchmark_rets = bench_value.pct_change()
    benchmark_rets.iloc[first_trade_idx] = 0.0

    print(pf.returns_stats(benchmark_rets=benchmark_rets))
    pf.returns().vbt.returns(benchmark_rets=benchmark_rets).plot_cumulative().show()

    return pf

## Run Experiments

Execute the two cells below to compare Black-Litterman vs Markowitz against SPY.

In [ ]:
print('=== Black-Litterman ===')
pf_bl = backtesting(model=black_litterman_opt)

In [ ]:
print('=== Markowitz (classical) ===')
pf_mw = backtesting(model=markowitz_opt)